In [1]:
import torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)


In [3]:

def vocab_expansion(base_model, tokenizer ,src_lang):

        tokenizer.add_special_tokens({"additional_special_tokens": [src_lang]})
        base_model.resize_token_embeddings(len(tokenizer))
        # Only set lang_code_to_id if the slow tokenizer supports it
        tokenizer.lang_code_to_id[src_lang] = tokenizer.convert_tokens_to_ids(src_lang)

        base_model.config.vocab_size = len(tokenizer)

        # vocab expansion
        src_id = tokenizer.convert_tokens_to_ids("ar_AR")
        tgt_id = tokenizer.convert_tokens_to_ids(src_lang)
        with torch.no_grad():
            base_model.model.shared.weight[tgt_id] = base_model.model.shared.weight[
                src_id
            ].clone()


In [4]:
tokenizer = AutoTokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
base_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

vocab_expansion(base_model, tokenizer, src_lang="ak_XX")

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [5]:

lora_config = LoraConfig(
    init_lora_weights="gaussian",
    r=32,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type=TaskType.SEQ_2_SEQ_LM,
    # Target attention projection layers
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    modules_to_save=["embed_tokens", "shared"],
    ensure_weight_tying=True,
    # trainable_token_indices MAYBE USE THIS
)


In [7]:
model = get_peft_model(base_model, lora_config)

In [8]:
merged_model = model.merge_and_unload()
merged_model.save_pretrained("mbart-lora-r32-alpha-16")
tokenizer.save_pretrained("mbart-lora-r32-alpha-16")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('mbart-lora-r32-alpha-16/tokenizer_config.json',
 'mbart-lora-r32-alpha-16/tokenizer.json')

In [ ]:
## KILL YOUR KERNEL AND TRY TO LOAD THEM

In [20]:
tokenizer = AutoTokenizer.from_pretrained("mbart-lora-r32-alpha-16")
model = AutoModelForSeq2SeqLM.from_pretrained("mbart-lora-r32-alpha-16")

Loading weights:   0%|          | 0/518 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [21]:
assert model.config.vocab_size == len(tokenizer)

In [18]:
!hf auth whoami

user:  MarkSpaghetti


In [22]:
model.push_to_hub("MarkSpaghetti/mbart-lora-r32-alpha-16")
tokenizer.push_to_hub("MarkSpaghetti/mbart-lora-r32-alpha-16")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/MarkSpaghetti/mbart-lora-r32-alpha-16/commit/5f652928b07d12bf61032780cf511fbe2ccce042', commit_message='Upload tokenizer', commit_description='', oid='5f652928b07d12bf61032780cf511fbe2ccce042', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MarkSpaghetti/mbart-lora-r32-alpha-16', endpoint='https://huggingface.co', repo_type='model', repo_id='MarkSpaghetti/mbart-lora-r32-alpha-16'), pr_revision=None, pr_num=None)

In [2]:
## KILL THE KERNEL AND TRY TO LOAD IT

tokenizer = AutoTokenizer.from_pretrained("MarkSpaghetti/mbart-lora-r32-alpha-16")
model = AutoModelForSeq2SeqLM.from_pretrained("MarkSpaghetti/mbart-lora-r32-alpha-16")


Loading weights:   0%|          | 0/518 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [3]:
assert model.config.vocab_size == len(tokenizer)

TokenizersBackend(name_or_path='MarkSpaghetti/mbart-lora-r32-alpha-16', vocab_size=250054, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	250001: AddedToken("ar_AR", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	250002: AddedToken("cs_CZ", rstrip=False, lstrip=False, single_word=False, normalized=False, specia